# Лабораторна робота — Урок 29: SQL основи

> Всі завдання виконуються з бібліотекою `sqlite3`.
> Починайте кожне завдання з порожньої бази даних (або у вбудованій `:memory:`).

---

## Сценарій: Бібліотечна система

Ви проєктуєте базу даних для **публічної бібліотеки**. В бібліотеці є читачі, книги, автори та записи про позичання книг.

```
authors ──────────── books ──────────── borrows
(1 автор = багато книг)    (1 книга = багато позичань)
                   members ─────────────┘
                   (1 читач = багато позичань)
```

In [ ]:
import sqlite3

# Ініціалізуємо базу даних та вмикаємо foreign keys
conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row
conn.execute("PRAGMA foreign_keys = ON")
cursor = conn.cursor()
print('База даних готова!')

---
## Завдання 1 — DDL: Спроєктуйте схему (⭐)

Створіть 4 таблиці з правильними обмеженнями:

**`authors`**: `author_id` (PK), `full_name` (NOT NULL), `country` (NOT NULL)

**`books`**: `book_id` (PK), `title` (NOT NULL), `year_published` (INT), `author_id` (FK → authors), `copies_total` (INT DEFAULT 1)

**`members`**: `member_id` (PK), `full_name` (NOT NULL), `email` (UNIQUE, NOT NULL), `joined_date` (TEXT)

**`borrows`**: `borrow_id` (PK), `book_id` (FK → books, ON DELETE RESTRICT), `member_id` (FK → members, ON DELETE RESTRICT), `borrow_date` (NOT NULL), `return_date` (TEXT, може бути NULL якщо ще не повернули)

In [ ]:
# TODO: Створіть таблиці authors, books, members, borrows
# Кожна таблиця — окремий cursor.execute()
# Не забудьте conn.commit() після всіх CREATE TABLE

# --- ВАШЕ РІШЕННЯ ---





# Перевірка: після виконання цей код має вивести 4 таблиці
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = [row['name'] for row in cursor.fetchall()]
print('Таблиці в базі:', tables)
assert set(tables) == {'authors', 'books', 'borrows', 'members'}, 'Деякі таблиці відсутні!'
print('✓ Завдання 1 виконано!')

---
## Завдання 2 — DML: Заповніть базу даними (⭐)

Вставте такі дані:

**Автори (мінімум 4):**
- Ліна Костенко (Україна)
- Іван Франко (Україна)
- Джордж Орвелл (Велика Британія)
- Харукі Муракамі (Японія)

**Книги (мінімум 6, як мінімум 2 від одного автора):**
- Від Ліни Костенко: "Маруся Чурай" (1979, 5 примірників), "Берестечко" (1999, 3 примірники)
- Від Івана Франка: "Захар Беркут" (1883, 4 примірники)
- Від Орвелла: "1984" (1949, 7 примірників), "Скотоферма" (1945, 6 примірників)
- Від Муракамі: "Норвезький ліс" (1987, 5 примірників)

**Члени бібліотеки (мінімум 4):** довільні

**Позичання (мінімум 5):** довільні (мінімум 2 — ще не повернуто, `return_date = NULL`)

In [ ]:
# TODO: Вставте всі дані за допомогою executemany() де можливо
# Пам'ятайте: спочатку батьківські таблиці (authors, members), потім дочірні (books, borrows)

# --- ВАШЕ РІШЕННЯ ---





# Перевірка
for table in ['authors', 'books', 'members', 'borrows']:
    cursor.execute(f'SELECT COUNT(*) AS cnt FROM {table}')
    count = cursor.fetchone()['cnt']
    print(f'  {table}: {count} записів')
    assert count >= {'authors': 4, 'books': 6, 'members': 4, 'borrows': 5}[table], \
        f'Недостатньо записів у {table}!'
print('✓ Завдання 2 виконано!')

---
## Завдання 3 — DML: UPDATE та DELETE (⭐)

a) Оновіть кількість примірників книги "1984" — збільшіть на 2 (без hard-коду нового значення, через `+2`)

b) Позначте одне з позичань як повернуте: встановіть `return_date = '2025-05-01'` для позичання, де `return_date IS NULL` (використайте `LIMIT 1`)

c) Додайте тестового члена і потім видаліть його (перевірте `rowcount`)

In [ ]:
# TODO: a) Оновіть кількість примірників книги '1984'


# TODO: b) Оновіть return_date для одного неповерненого позичання


# TODO: c) Вставте тестового члена і видаліть його


# --- ВАШЕ РІШЕННЯ ---


conn.commit()

# Перевірка a)
cursor.execute("SELECT copies_total FROM books WHERE title = '1984'")
row = cursor.fetchone()
assert row is not None, 'Книгу 1984 не знайдено!'
assert row['copies_total'] == 9, f'Очікувалось 9, отримано {row["copies_total"]}'
print('✓ a) Кількість примірників оновлено!')

# Перевірка b)
cursor.execute("SELECT COUNT(*) AS cnt FROM borrows WHERE return_date IS NOT NULL")
assert cursor.fetchone()['cnt'] >= 1, 'Жодне позичання не оновлено!'
print('✓ b) Повернення зафіксовано!')

print('✓ Завдання 3 виконано!')

---
## Завдання 4 — DQL: Запити з фільтрацією та агрегацією (⭐⭐)

Напишіть 4 окремі SELECT-запити:

a) Всі книги, опубліковані після 1980 року, відсортовані від новіших до старіших

b) Кількість книг по кожному автору (`author_id`, `books_count`) — відсортуйте за `books_count` DESC

c) Члени бібліотеки, які ще мають непові`рнені книги (де `return_date IS NULL`) — без дублікатів

d) Топ-3 найстаріші книги (за `year_published`)

In [ ]:
# TODO a): Книги після 1980 року
print('=== a) Книги після 1980 ===')
# --- ВАШЕ РІШЕННЯ ---



print()

# TODO b): Кількість книг по авторах
print('=== b) Книги по авторах ===')
# --- ВАШЕ РІШЕННЯ ---



print()

# TODO c): Члени з неповерненими книгами (DISTINCT!)
print('=== c) Члени з неповерненими книгами ===')
# --- ВАШЕ РІШЕННЯ ---



print()

# TODO d): Топ-3 найстаріші книги
print('=== d) Топ-3 найстаріші книги ===')
# --- ВАШЕ РІШЕННЯ ---


---
## Завдання 5 — JOIN: З'єднання таблиць (⭐⭐)

a) **INNER JOIN**: Виведіть список всіх позичань з назвою книги та ім'ям члена бібліотеки (3 таблиці: `borrows`, `books`, `members`)

b) **LEFT JOIN**: Виведіть всіх авторів та кількість їхніх книг у бібліотеці (включно з авторами без книг, якщо такі є)

c) **LEFT JOIN + фільтр NULL**: Знайдіть книги, які **жодного разу не позичали** (hint: LEFT JOIN з `borrows`, де `borrow_id IS NULL`)

In [ ]:
# TODO a): INNER JOIN — позичання з деталями
print('=== a) Активні позичання ===')
# --- ВАШЕ РІШЕННЯ ---



print()

# TODO b): LEFT JOIN — автори і їхня кількість книг
print('=== b) Автори та кількість книг ===')
# --- ВАШЕ РІШЕННЯ ---



print()

# TODO c): Книги, які ніколи не позичали
print('=== c) Книги без позичань ===')
# --- ВАШЕ РІШЕННЯ ---


---
## Завдання 6 — TCL: Транзакція з ROLLBACK (⭐⭐⭐)

Напишіть функцію `borrow_book(conn, member_id, book_id, date)` яка:

1. Відкриває транзакцію (`BEGIN`)
2. Перевіряє, чи є вільні примірники книги (`copies_total > 0`) — якщо ні, кидає `ValueError`
3. Зменшує `copies_total` на 1
4. Вставляє запис у таблицю `borrows`
5. Робить `COMMIT` — або `ROLLBACK` при помилці

Протестуйте:
- Успішне позичання
- Спробу позичити книгу з 0 примірниками (має спрацювати ROLLBACK)

In [ ]:
def borrow_book(conn, member_id: int, book_id: int, borrow_date: str):
    """
    Атомарна операція позичання книги.
    Або всі зміни збережені — або жодної.
    """
    cursor = conn.cursor()

    # TODO: Реалізуйте функцію згідно з описом
    # --- ВАШЕ РІШЕННЯ ---
    pass


# Тест 1: Успішне позичання
# borrow_book(conn, member_id=1, book_id=1, borrow_date='2025-05-01')

# Тест 2: Книга з 0 примірниками
# Спочатку встановіть copies_total = 0 для якоїсь книги, потім спробуйте позичити


---
## Завдання 7 — Бонус: Підзапити (⭐⭐⭐)

a) Знайдіть автора, чиї книги **в середньому найновіші** (найбільший середній `year_published`). Використайте підзапит або `GROUP BY` + `ORDER BY LIMIT 1`.

b) Знайдіть членів бібліотеки, які позичали книги **більше одного разу** (через `GROUP BY` + `HAVING` або підзапит із `COUNT`).

c) **DDL + DML комбо**: Додайте до таблиці `books` колонку `genre` (TEXT DEFAULT 'Невизначено') і оновіть жанри для всіх книг (довільно).

In [ ]:
# TODO a): Автор із найновішими книгами в середньому
print('=== a) Автор з найновішими книгами ===')
# --- ВАШЕ РІШЕННЯ ---


print()

# TODO b): Члени, які позичали більше одного разу
print('=== b) Активні читачі ===')
# --- ВАШЕ РІШЕННЯ ---


print()

# TODO c): ALTER TABLE + UPDATE
print('=== c) Додавання жанрів ===')
# --- ВАШЕ РІШЕННЯ ---


---
## Таблиця оцінювання

| Завдання | Бали | Ключові навички |
|----------|------|-----------------|
| 1 — DDL (CREATE TABLE) | 15 | PRIMARY KEY, FOREIGN KEY, обмеження |
| 2 — DML (INSERT) | 15 | executemany, порядок вставки |
| 3 — DML (UPDATE/DELETE) | 15 | WHERE, rowcount, lastrowid |
| 4 — DQL (SELECT) | 20 | GROUP BY, HAVING, ORDER BY, DISTINCT |
| 5 — JOIN | 20 | INNER JOIN, LEFT JOIN, NULL-фільтрація |
| 6 — TCL (транзакції) | 15 | BEGIN/COMMIT/ROLLBACK, обробка помилок |
| **Бонус** — підзапити | +10 | Subqueries, ALTER TABLE |
| **Разом** | **100** | |

---
*Урок 29 | Модуль 3 | Python Course*